# Generating Embedding for Query Image

In this notebook, we will:

- Preprocess a query image from the matches dataset.
- Generate its embedding using the PLIP model.
- Save the query embedding to disk.
- Clear memory after processing.

---

## **Step 1: Import Libraries**

---

In [1]:
import os
import torch
import numpy as np
from transformers import CLIPModel, CLIPProcessor
from PIL import Image, ImageEnhance
import re
import random

## **Step 2: Set Up Device**

---

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


## **Step 3: Define Helper Functions**

---

In [3]:
def preprocess_image(image_path, size=(224, 224), contrast_factor=1.5):
    img = Image.open(image_path).convert("RGB")
    img = img.resize(size)
    enhancer = ImageEnhance.Contrast(img)
    img = enhancer.enhance(contrast_factor)
    return img

## **Step 4: Load PLIP Model and Processor**

---

In [4]:
# Load PLIP model and processor
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

/Users/mohammedkhodorfirasal-tal/Documents/Professional/Work/Fellowship - Novartis/Data/venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


## **Step 5: Select and Preprocess Query Image**

---

In [5]:
# Define matches directory
matches_dir = './matches/'

# Get list of images in the matches directory
matches_image_files = [f for f in os.listdir(matches_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]

# Function to extract base ID from filename
def extract_base_id(filename):
    match = re.match(r'(\d+)[_.]\d+\.\w+', filename)
    if match:
        return match.group(1)
    else:
        return None

# Create a dictionary to map base IDs to their corresponding image filenames
matches_dict = {}
for filename in matches_image_files:
    base_id = extract_base_id(filename)
    if base_id:
        matches_dict.setdefault(base_id, []).append(filename)

# Select a random base ID and query image
random_base_id = random.choice(list(matches_dict.keys()))
matching_images = matches_dict[random_base_id]
query_image_filename = random.choice(matching_images)
query_image_path = os.path.join(matches_dir, query_image_filename)

# Save other matching images (for later use)
other_matching_images = [img for img in matching_images if img != query_image_filename]
np.save('other_matching_images.npy', other_matching_images)
np.save('query_image_info.npy', np.array([query_image_filename, query_image_path]))

print(f"Selected query image: {query_image_filename}")
print(f"Other matching images: {other_matching_images}")

Selected query image: 722_04.jpg
Other matching images: ['722_01.jpg', '722_03.jpg']


## **Step 6: Generate Query Embedding**

---

In [6]:
# Preprocess the query image
query_image = preprocess_image(query_image_path)

# Generate and normalize embedding for the query image
with torch.no_grad():
    inputs = processor(images=query_image, return_tensors="pt").to(device)
    query_embedding = model.get_image_features(**inputs)
    query_embedding = query_embedding.cpu().numpy()
    # Normalize the query embedding
    query_embedding = query_embedding / np.linalg.norm(query_embedding)

np.save('query_embedding.npy', query_embedding)

print("Query embedding generated and saved.")

Query embedding generated and saved.


## **Step 7: Clean Up**

---

In [7]:
# Free up memory
del model, processor, query_image, inputs, query_embedding
torch.cuda.empty_cache()
import gc
gc.collect()

8